# Exp 17 — Глубокий EDA русских датасетов

**Цели:**
1. Классифицировать колонки по «естественности» (ID-like / категориальные / текст / числовые) — чтобы понять, что реально использовать как ER-фичу
2. Найти within-dataset дубликаты для **всех 6** датасетов
3. Cross-dataset overlap **для всех релевантных пар** (cars_ru × auto_ru × auto_ru_2020 — все три про авто!)
4. Multi-key matching: проверять не только sell_id, но и (brand, model, year, mileage), описания, URL и т.д.

In [ ]:
from __future__ import annotations
import re
import sys
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE
for p in [HERE, *HERE.parents]:
    if (p / 'pyproject.toml').exists():
        ROOT = p
        break
RAW_RU = ROOT / 'data' / 'raw_ru'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 100)

print('ROOT:', ROOT)

In [ ]:
dfs = {}
for name in ('lamoda', 'cars_ru', 'ozon', 'auto_ru', 'auto_ru_2020', 'devices'):
    p = RAW_RU / name / 'clean.parquet'
    if p.exists():
        dfs[name] = pd.read_parquet(p)
        print(f'{name:15s}  {dfs[name].shape}')
    else:
        print(f'{name:15s}  NOT FOUND')

---
## 1. Классификация колонок (информационная — НЕ для фильтрации)

**Важно:** на ER модель идут **все** колонки. GATv2 с attention сам выучит, какие важны, а какие игнорировать. Эта классификация нужна для:
- Выбора **блокинг-ключей** — для них как раз хороши `id_like` / категориальные с разумной кардинальностью
- Выбора **полей для синтетической порчи** (`value_corruption`) — категориальные/текстовые
- Понимания, **что меняется между дубликатами** — текст/числа варьируются, ID — нет

Условные типы:
- **id_like** — почти уникальное значение на строку (URL, sell_id). Идеальный блокинг-ключ для within-dataset.
- **constant** — n_unique ≤ 1 (после `_drop_dead` таких быть не должно)
- **binary** — bool или 2 значения
- **categorical** — кардинальность 3..1000, заполненность > 20%
- **high_card** — > 1000 значений, но не уникальная
- **numeric** / **text** / **datetime**

In [ ]:
ID_HINT_RE = re.compile(r'(^|[._])(id|url|link|image|hash|guid|uuid|src|sell_id|timestamp|unixtime|date|артикул|ссылка)([._]|$)', re.IGNORECASE)

def classify_column(s: pd.Series, col: str, n_rows: int) -> tuple[str, dict]:
    info: dict = {}
    fill = 1 - s.isna().mean()
    info['fill'] = fill

    if pd.api.types.is_datetime64_any_dtype(s):
        try:
            info['n_unique'] = int(s.nunique(dropna=True))
        except TypeError:
            info['n_unique'] = -1
        return 'datetime', info

    if pd.api.types.is_bool_dtype(s):
        info['n_unique'] = 2
        return 'binary', info

    try:
        nu = int(s.nunique(dropna=True))
    except TypeError:
        info['n_unique'] = -1
        return 'list_dict', info
    info['n_unique'] = nu

    if nu <= 1:
        return 'constant', info

    if pd.api.types.is_numeric_dtype(s):
        if nu == 2:
            return 'binary', info
        return 'numeric', info

    # object / string columns
    non_null = s.dropna()
    if len(non_null) == 0:
        return 'constant', info
    sample = non_null.head(1000).astype(str)
    avg_len = sample.str.len().mean()
    info['avg_len'] = avg_len

    # ID-like: имя похоже на ID или почти уникально + значение короткое/символьное
    uniqueness = nu / max(len(non_null), 1)
    info['uniqueness'] = uniqueness
    if ID_HINT_RE.search(col) or uniqueness > 0.95:
        return 'id_like', info

    if avg_len > 80:
        return 'text', info

    if nu == 2:
        return 'binary', info
    if nu <= 1000:
        return 'categorical', info
    return 'high_card', info

def classify_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(df)
    for col in df.columns:
        kind, info = classify_column(df[col], col, n)
        rows.append({
            'column': col,
            'dtype': str(df[col].dtype),
            'kind': kind,
            'n_unique': info.get('n_unique', -1),
            'fill_%': round(info.get('fill', 0) * 100, 1),
            'uniq_ratio': round(info.get('uniqueness', 0), 3),
            'avg_len': round(info.get('avg_len', 0), 1) if 'avg_len' in info else None,
        })
    return pd.DataFrame(rows)

In [ ]:
classifications = {name: classify_dataframe(df) for name, df in dfs.items()}

# Сводка по типам колонок
print('Распределение типов колонок по датасетам:\n')
summary = pd.DataFrame({
    name: c['kind'].value_counts() for name, c in classifications.items()
}).fillna(0).astype(int)
print(summary.to_string())

In [ ]:
# id_like колонки — кандидаты для БЛОКИНГ-КЛЮЧЕЙ (на ER их всё равно подаём, GAT разберётся).
# Внутри одного датасета они почти уникальны — отлично для блокинга within-dataset
# (например, group by URL у Ozon).
for name, c in classifications.items():
    bad = c[c['kind'].isin(['id_like', 'list_dict'])]
    if len(bad):
        print(f'\n--- {name}: id_like / list-колонки ({len(bad)}) — кандидаты для блокинга ---')
        print(bad[['column', 'kind', 'n_unique', 'uniq_ratio', 'fill_%']].to_string(index=False))

In [ ]:
# Обзор «семантических» полей — категориальные/численные/текстовые/datetime с разумной заполненностью.
# Это поля, которые точно несут полезный сигнал для ER (хотя на вход модели идут ВСЕ колонки —
# GATv2 сам взвесит важность).
GOOD_KINDS = {'categorical', 'numeric', 'text', 'binary', 'high_card', 'datetime'}

for name, c in classifications.items():
    good = c[c['kind'].isin(GOOD_KINDS) & (c['fill_%'] >= 20)]
    print(f'\n--- {name}: семантически нагруженные поля (fill ≥ 20%, всего {len(good)}) ---')
    print(good.sort_values(['kind', 'fill_%'], ascending=[True, False])[['column', 'kind', 'n_unique', 'fill_%', 'avg_len']].head(30).to_string(index=False))

---
## 2. Within-dataset дубликаты для ВСЕХ датасетов

Для каждого датасета пробуем разные ключи блокинга и считаем число пар-кандидатов.

In [ ]:
def blocking_stats(df: pd.DataFrame, key_cols: list[str], label: str = '') -> dict:
    """Считает: блоков всего, блоков с >1 строкой, всего пар, размер крупнейшего блока."""
    if not all(c in df.columns for c in key_cols):
        return {'label': label, 'note': 'columns missing'}
    keys = df[key_cols].fillna('?').astype(str).agg('|'.join, axis=1)
    sizes = keys.value_counts()
    multi = sizes[sizes > 1]
    n_pairs = int((multi * (multi - 1) // 2).sum())
    return {
        'label': label or '+'.join(key_cols),
        'n_blocks': int(len(sizes)),
        'n_multi_blocks': int(len(multi)),
        'n_pairs': n_pairs,
        'max_block': int(sizes.max()),
        'mean_multi': round(multi.mean(), 2) if len(multi) else 0,
    }

def show_blocking_results(results: list[dict]) -> None:
    print(pd.DataFrame(results).to_string(index=False))

In [ ]:
print('=== lamoda ===')
lam = dfs['lamoda']
show_blocking_results([
    blocking_stats(lam, ['Title', 'Brand'], 'Title+Brand'),
    blocking_stats(lam, ['Title', 'Brand', 'Price'], 'Title+Brand+Price'),
    blocking_stats(lam, ['about.Артикул'], 'Артикул'),
    blocking_stats(lam, ['Brand', 'about.Цвет'], 'Brand+Цвет'),
    blocking_stats(lam, ['Title'], 'Title'),
])

In [ ]:
print('=== cars_ru ===')
cars = dfs['cars_ru'].copy()
cars['_km_bucket'] = (cars['km_age'].fillna(-1) // 10000 * 10000).astype('Int64').astype(str)
cars['_price_bucket'] = (cars['price_rub'].fillna(-1) // 50000 * 50000).astype('Int64').astype(str)

show_blocking_results([
    blocking_stats(cars, ['mark', 'model'], 'mark+model'),
    blocking_stats(cars, ['mark', 'model', 'year'], 'mark+model+year'),
    blocking_stats(cars, ['mark', 'model', 'year', '_km_bucket'], 'mark+model+year+km10k'),
    blocking_stats(cars, ['mark', 'model', 'year', 'color'], 'mark+model+year+color'),
    blocking_stats(cars, ['mark', 'model', 'year', '_km_bucket', '_price_bucket'], 'mark+model+year+km+price'),
    blocking_stats(cars, ['mark', 'model', 'generation', 'year'], 'mark+model+gen+year'),
])

In [ ]:
# Дополнительно для cars_ru: дубли description (одинаковое объявление перепостят)
if 'description' in cars.columns:
    desc_counts = cars['description'].dropna().value_counts()
    print(f'Уникальных description: {desc_counts.nunique():,}')
    print(f'Description c >1 объявлением (точные дубли текста): {(desc_counts > 1).sum():,}')
    print(f'Доля точных дублей: {(desc_counts > 1).sum() / len(desc_counts) * 100:.1f}%')
    print('\nТоп-5 повторяющихся описаний:')
    for desc, cnt in desc_counts.head(5).items():
        if cnt > 1:
            print(f'  ({cnt}×) {desc[:120]!r}')

In [ ]:
print('=== ozon ===')
ozon = dfs['ozon']
url_col = 'Ссылка на товар'
src_col = '__source_file'

# Внутри одного файла URL должны быть уникальны? Проверим.
in_file_dups = ozon.groupby([src_col, url_col]).size()
print(f'URL внутри одного файла встречается >1 раз: {(in_file_dups > 1).sum():,} случаев')

show_blocking_results([
    blocking_stats(ozon, [url_col], 'URL'),
    blocking_stats(ozon, ['Название товара', 'Бренд'], 'Название+Бренд'),
    blocking_stats(ozon, ['Название товара'], 'Название'),
    blocking_stats(ozon, ['Бренд', 'Категория 4 уровня'], 'Бренд+Кат4'),
])

# URL пересечения по файлам
url_in_files = ozon.groupby(url_col)[src_col].nunique()
print(f'\nURL присутствует в N файлах:')
print(url_in_files.value_counts().sort_index().head(15).to_string())

In [ ]:
print('=== auto_ru ===')
ar = dfs['auto_ru'].copy()
ar['_km_bucket'] = (ar['mileage'].fillna(-1) // 10000 * 10000).astype('Int64').astype(str)
ar['_price_bucket'] = (ar['price'].fillna(-1) // 50000 * 50000).astype('Int64').astype(str)

# Внутри auto_ru тоже могут быть дубликаты — одно объявление перепостили
show_blocking_results([
    blocking_stats(ar, ['sell_id'], 'sell_id'),
    blocking_stats(ar, ['car_url'], 'car_url'),
    blocking_stats(ar, ['brand', 'model_name', 'productionDate'], 'brand+model+year'),
    blocking_stats(ar, ['brand', 'model_name', 'productionDate', '_km_bucket'], 'brand+model+year+km10k'),
    blocking_stats(ar, ['brand', 'model_name', 'productionDate', '_km_bucket', 'color'], 'brand+model+year+km+color'),
    blocking_stats(ar, ['vehicleConfiguration', 'brand', 'productionDate'], 'config+brand+year'),
    blocking_stats(ar, ['description'], 'description (exact dup)'),
])

In [ ]:
print('=== auto_ru_2020 ===')
ar2 = dfs['auto_ru_2020'].copy()
ar2['_km_bucket'] = (ar2['mileage'].fillna(-1) // 10000 * 10000).astype('Int64').astype(str)

show_blocking_results([
    blocking_stats(ar2, ['sell_id'], 'sell_id'),
    blocking_stats(ar2, ['brand', 'model_name', 'productionDate'], 'brand+model+year'),
    blocking_stats(ar2, ['brand', 'model_name', 'productionDate', '_km_bucket'], 'brand+model+year+km10k'),
    blocking_stats(ar2, ['brand', 'model_name', 'productionDate', '_km_bucket', 'color'], 'brand+model+year+km+color'),
    blocking_stats(ar2, ['description'], 'description'),
    blocking_stats(ar2, ['vehicleConfiguration', 'brand', 'productionDate'], 'config+brand+year'),
])

In [ ]:
print('=== devices ===')
dev = dfs['devices']
show_blocking_results([
    blocking_stats(dev, ['device_model'], 'device_model'),
    blocking_stats(dev, ['device_model', 'manufacturer'], 'device_model+manufacturer'),
    blocking_stats(dev, ['device_model', 'manufacturer', 'release_year'], 'device_model+mfr+year'),
    blocking_stats(dev, ['model_family', 'manufacturer', 'device_type'], 'family+mfr+type'),
    blocking_stats(dev, ['manufacturer', 'device_type', 'release_year'], 'mfr+type+year'),
])

---
## 3. Cross-dataset overlap: cars_ru × auto_ru × auto_ru_2020

Все три — про русский авторынок. Возможны прямые матчи.

In [ ]:
# Какие колонки есть в каких датасетах? Сначала покажем «сырые» названия.
print('cars_ru     ', sorted(dfs['cars_ru'].columns.tolist()))
print('\nauto_ru     ', sorted(dfs['auto_ru'].columns.tolist())[:40], '...')
print('\nauto_ru_2020', sorted(dfs['auto_ru_2020'].columns.tolist())[:40], '...')

In [ ]:
# Семантическое выравнивание полей между тремя датасетами авто
field_map = pd.DataFrame([
    {'concept': 'brand',     'cars_ru': 'mark',      'auto_ru': 'brand',        'auto_ru_2020': 'brand'},
    {'concept': 'model',     'cars_ru': 'model',     'auto_ru': 'model_name',   'auto_ru_2020': 'model_name'},
    {'concept': 'year',      'cars_ru': 'year',      'auto_ru': 'productionDate','auto_ru_2020': 'productionDate'},
    {'concept': 'mileage',   'cars_ru': 'km_age',    'auto_ru': 'mileage',      'auto_ru_2020': 'mileage'},
    {'concept': 'color',     'cars_ru': 'color',     'auto_ru': 'color',        'auto_ru_2020': 'color'},
    {'concept': 'body_type', 'cars_ru': 'body_type', 'auto_ru': 'bodyType',     'auto_ru_2020': 'bodyType'},
    {'concept': 'fuel',      'cars_ru': 'engine_type','auto_ru': 'fuelType',    'auto_ru_2020': 'fuelType'},
    {'concept': 'transmission','cars_ru': 'transmission','auto_ru': 'vehicleTransmission','auto_ru_2020': 'vehicleTransmission'},
    {'concept': 'description','cars_ru': 'description','auto_ru': 'description','auto_ru_2020': 'description'},
    {'concept': 'price',     'cars_ru': 'price_rub', 'auto_ru': 'price',        'auto_ru_2020': 'price'},
    {'concept': 'displacement','cars_ru': 'displacement','auto_ru': 'engineDisplacement','auto_ru_2020': 'engineDisplacement'},
    {'concept': 'horsepower','cars_ru': 'horse_power','auto_ru': 'enginePower', 'auto_ru_2020': 'enginePower'},
])
print(field_map.to_string(index=False))

In [ ]:
# Vocabulary overlap по брендам
def normalize(s):
    if pd.isna(s):
        return None
    return str(s).strip().lower()

brands_cars = set(dfs['cars_ru']['mark'].dropna().map(normalize))
brands_ar   = set(dfs['auto_ru']['brand'].dropna().map(normalize))
brands_ar2  = set(dfs['auto_ru_2020']['brand'].dropna().map(normalize))

print(f'Бренды в cars_ru:      {len(brands_cars):>4}  пример: {sorted(brands_cars)[:6]}')
print(f'Бренды в auto_ru:      {len(brands_ar):>4}  пример: {sorted(brands_ar)[:6]}')
print(f'Бренды в auto_ru_2020: {len(brands_ar2):>4}  пример: {sorted(brands_ar2)[:6]}')

print(f'\ncars_ru ∩ auto_ru:           {len(brands_cars & brands_ar)}')
print(f'cars_ru ∩ auto_ru_2020:      {len(brands_cars & brands_ar2)}')
print(f'auto_ru ∩ auto_ru_2020:      {len(brands_ar & brands_ar2)}')
print(f'все три:                     {len(brands_cars & brands_ar & brands_ar2)}')

only_in_cars_ru  = brands_cars - brands_ar - brands_ar2
print(f'\nБренды только в cars_ru ({len(only_in_cars_ru)}): {sorted(only_in_cars_ru)[:10]} ...')
print(f'Бренды только в auto_ru ({len(brands_ar - brands_cars - brands_ar2)}): {sorted(brands_ar - brands_cars - brands_ar2)[:10]}')

In [ ]:
# (brand, model) overlap — более жёсткая метрика
def get_brand_models(df, brand_col, model_col):
    return set(
        (normalize(b), normalize(m))
        for b, m in zip(df[brand_col], df[model_col])
        if pd.notna(b) and pd.notna(m)
    )

bm_cars = get_brand_models(dfs['cars_ru'], 'mark', 'model')
bm_ar   = get_brand_models(dfs['auto_ru'], 'brand', 'model_name')
bm_ar2  = get_brand_models(dfs['auto_ru_2020'], 'brand', 'model_name')

print(f'(brand, model) пар:')
print(f'  cars_ru:      {len(bm_cars):,}')
print(f'  auto_ru:      {len(bm_ar):,}')
print(f'  auto_ru_2020: {len(bm_ar2):,}')
print(f'\nПересечения:')
print(f'  cars_ru ∩ auto_ru:           {len(bm_cars & bm_ar):,}')
print(f'  cars_ru ∩ auto_ru_2020:      {len(bm_cars & bm_ar2):,}')
print(f'  auto_ru ∩ auto_ru_2020:      {len(bm_ar & bm_ar2):,}')
print(f'  все три:                     {len(bm_cars & bm_ar & bm_ar2):,}')

# Несовпадения часто из-за разной нотации (Rolls-Royce vs Rolls Royce)
print(f'\nПримеры (brand, model) только в cars_ru:')
print(list(bm_cars - bm_ar - bm_ar2)[:8])
print(f'\nПримеры в auto_ru, но не в cars_ru:')
print(list(bm_ar - bm_cars)[:8])

In [ ]:
# Cross-dataset blocking: сколько кандидатов даёт каждый ключ для каждой пары датасетов
def cross_block_stats(
    df_a: pd.DataFrame, df_b: pd.DataFrame,
    cols_a: list[str], cols_b: list[str],
    label: str,
) -> dict:
    def key(df, cols):
        return df[cols].fillna('?').astype(str).apply(lambda s: s.str.lower().str.strip()).agg('|'.join, axis=1)
    ka = key(df_a, cols_a)
    kb = key(df_b, cols_b)
    counts_a = ka.value_counts()
    counts_b = kb.value_counts()
    common = set(counts_a.index) & set(counts_b.index)
    n_pairs = sum(counts_a[k] * counts_b[k] for k in common)
    return {
        'pair': label,
        'common_blocks': len(common),
        'n_candidate_pairs': int(n_pairs),
        'avg_block_size_a': round(np.mean([counts_a[k] for k in common]), 1) if common else 0,
        'avg_block_size_b': round(np.mean([counts_b[k] for k in common]), 1) if common else 0,
    }

results = []
# Подготовим km buckets
for name, df in [('cars_ru', cars), ('auto_ru', ar), ('auto_ru_2020', ar2)]:
    pass  # уже есть _km_bucket выше

# 1) brand+model+year
results.append(cross_block_stats(cars, ar, ['mark', 'model', 'year'], ['brand', 'model_name', 'productionDate'], 'cars_ru × auto_ru: brand+model+year'))
results.append(cross_block_stats(cars, ar2, ['mark', 'model', 'year'], ['brand', 'model_name', 'productionDate'], 'cars_ru × auto_ru_2020: brand+model+year'))
results.append(cross_block_stats(ar, ar2, ['brand', 'model_name', 'productionDate'], ['brand', 'model_name', 'productionDate'], 'auto_ru × auto_ru_2020: brand+model+year'))

# 2) +mileage bucket
results.append(cross_block_stats(cars, ar, ['mark', 'model', 'year', '_km_bucket'], ['brand', 'model_name', 'productionDate', '_km_bucket'], 'cars_ru × auto_ru: +km10k'))
results.append(cross_block_stats(cars, ar2, ['mark', 'model', 'year', '_km_bucket'], ['brand', 'model_name', 'productionDate', '_km_bucket'], 'cars_ru × auto_ru_2020: +km10k'))
results.append(cross_block_stats(ar, ar2, ['brand', 'model_name', 'productionDate', '_km_bucket'], ['brand', 'model_name', 'productionDate', '_km_bucket'], 'auto_ru × auto_ru_2020: +km10k'))

# 3) +color
results.append(cross_block_stats(cars, ar, ['mark', 'model', 'year', '_km_bucket', 'color'], ['brand', 'model_name', 'productionDate', '_km_bucket', 'color'], 'cars_ru × auto_ru: +km10k+color'))
results.append(cross_block_stats(ar, ar2, ['brand', 'model_name', 'productionDate', '_km_bucket', 'color'], ['brand', 'model_name', 'productionDate', '_km_bucket', 'color'], 'auto_ru × auto_ru_2020: +km10k+color'))

print(pd.DataFrame(results).to_string(index=False))

In [ ]:
# Точные совпадения по сырым ID между датасетами
def s2set(s):
    return set(s.dropna().astype(str))

ids_ar  = s2set(ar['sell_id'])
ids_ar2 = s2set(ar2['sell_id'])
print(f'sell_id auto_ru ∩ auto_ru_2020: {len(ids_ar & ids_ar2):,}')

# car_url есть в auto_ru, в auto_ru_2020 нет — но всё равно проверим
if 'car_url' in ar.columns and 'car_url' in ar2.columns:
    urls_ar = s2set(ar['car_url'])
    urls_ar2 = s2set(ar2['car_url'])
    print(f'car_url auto_ru ∩ auto_ru_2020: {len(urls_ar & urls_ar2):,}')

# description дубли между датасетами (одинаковый текст объявления)
for left_name, left_df in [('cars_ru', dfs['cars_ru']), ('auto_ru', ar)]:
    for right_name, right_df in [('auto_ru', ar), ('auto_ru_2020', ar2)]:
        if left_name >= right_name:
            continue
        left_desc = s2set(left_df['description'])
        right_desc = s2set(right_df['description'])
        common = left_desc & right_desc
        print(f'description {left_name} ∩ {right_name}: {len(common):,}')

---
## 4. Внутри-датасетная согласованность значений

Например, бренд в cars_ru и auto_ru записан в разных регистрах: 'Kia' vs 'KIA'. Это влияет на блокинг.

In [ ]:
# Сколько брендов меняют написание после нормализации
for name, col in [('cars_ru', 'mark'), ('auto_ru', 'brand'), ('auto_ru_2020', 'brand')]:
    raw = dfs[name][col].dropna().astype(str)
    print(f'{name}.{col}: уникальных raw={raw.nunique()}, после lower/strip={raw.str.strip().str.lower().nunique()}')
    # Покажем те, что отличаются только регистром
    grouped = raw.groupby(raw.str.strip().str.lower()).agg(lambda x: list(set(x)))
    multi_form = grouped[grouped.map(len) > 1]
    if len(multi_form):
        print(f'  Бренды с несколькими формами:')
        for k, v in multi_form.head(5).items():
            print(f'    {k!r} -> {v}')

In [ ]:
# Проверим cars_ru body_type vs auto_ru bodyType
print('cars_ru body_type sample:')
print(dfs['cars_ru']['body_type'].value_counts().head(10).to_string())
print('\nauto_ru bodyType sample:')
print(ar['bodyType'].value_counts().head(10).to_string())
print('\nauto_ru_2020 bodyType sample:')
print(ar2['bodyType'].value_counts().head(10).to_string())

In [ ]:
# Топливо: cars_ru.engine_type vs auto_ru.fuelType vs auto_ru_2020.fuelType
print('cars_ru.engine_type:', dfs['cars_ru']['engine_type'].value_counts().to_dict())
print('auto_ru.fuelType:', ar['fuelType'].value_counts().to_dict())
print('auto_ru_2020.fuelType:', ar2['fuelType'].value_counts().to_dict())

---
## 5. Разнообразие значений — для синтетической порчи (`value_corruption`)

In [ ]:
# Категориальные поля с разумной кардинальностью + хорошей заполненностью —
# именно их хорошо «портить» (заменять на синонимы / вариации регистра).
# Числовые с большой дисперсией — кандидаты на format/noise corruption.

for name, c in classifications.items():
    print(f'\n=== {name} ===')
    cat = c[(c['kind'] == 'categorical') & (c['fill_%'] > 30) & (c['n_unique'] <= 100)]
    print(f'Категориальные fill>30%, n_unique≤100 ({len(cat)}):')
    print(cat[['column', 'n_unique', 'fill_%']].head(15).to_string(index=False))

In [ ]:
# Текстовые поля — естественные кандидаты на typo corruption
for name, c in classifications.items():
    text_cols = c[c['kind'] == 'text']
    if len(text_cols):
        print(f'{name}: text-колонок {len(text_cols)}: {text_cols["column"].tolist()}')
        for col in text_cols['column']:
            avg_len = c.loc[c['column'] == col, 'avg_len'].iloc[0]
            print(f'  {col}: avg_len={avg_len}')

---
## 6. Итог: рекомендации по разметке

После прогона всех ячеек выше у нас будут конкретные числа: сколько кандидат-пар даёт каждый блокинг-ключ, какой overlap между датасетами авто.

**Шаблон решения по каждому датасету:**

| Источник меток | Когда применять | Стоимость |
|---|---|---|
| Прямое совпадение ID/URL | id_like overlap > 0 | бесплатно |
| Группа по `(Title+Brand)` или `(brand+model+year+km)` | within-dataset, размер пар разумный | бесплатно (но нужна валидация) |
| Блокинг + LLM-агент (exp 13) | большой n_pairs, нет естественных ключей | дорого, но точно |
| Synthetic corruption | для расширения позитивов | бесплатно, но не «реальные» дубликаты |

Конкретный план — формируем после запуска всех ячеек.